In [ ]:
# 实验四：频繁项集与关联规则挖掘
# Task 1a: 读取数据集并创建status列
import pandas as pd
import json
from collections import Counter
from itertools import combinations

records = []
with open('iclr_2026_data.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

df = pd.DataFrame(records)
print(f'原始数据集包含 {len(df)} 篇论文')
print(f'列名: {list(df.columns)}')

def get_status(decision):
    """创建status列：已(条件)接收=1, 拒稿=0, 撤稿/桌拒=NaN"""
    if not decision or decision.strip() == '':
        return None
    if 'Accept' in decision:
        return 1
    elif decision == 'Reject':
        return 0
    else:
        return None

df['status'] = df['decision'].apply(get_status)
print(f'\nstatus列分布:')
print(df['status'].value_counts(dropna=False))
print(f'\n前5行数据预览:')
df[['title', 'keywords', 'decision', 'status']].head()

In [ ]:
# Task 1b: 删除空值记录，仅保留keywords和status，展示统计信息
# 删除status为空值的记录
df_clean = df.dropna(subset=['status']).copy()
df_clean['status'] = df_clean['status'].astype(int)

# 仅保留keywords和status两列
df_kw = df_clean[['keywords', 'status']].reset_index(drop=True)

n_papers = len(df_kw)
n_accept = df_kw['status'].sum()
accept_ratio = n_accept / n_papers

print(f'筛选后论文数量: {n_papers}')
print(f'已接收论文数量: {n_accept}')
print(f'已接收论文占比: {accept_ratio:.4f} ({accept_ratio*100:.2f}%)')
print(f'\n各状态分布:')
print(df_kw['status'].value_counts())
df_kw.head()


筛选后论文数量: 14175
已接收论文数量: 5359
已接收论文占比: 0.3781 (37.81%)

各状态分布:
status
0    8816
1    5359
Name: count, dtype: int64


,keywords,status
0,Emotion Detection; Commit Messages; Software E...,0
1,Multilingual LLMs; multilinguality; cross-ling...,0
2,Reinforcement Learning; Agent Alignment,0
3,Meta-Reinforcement Learning,0
4,omni-multimodal large language models; identit...,0


In [3]:
# Task 1c: 关键词清洗、同义词合并、统计top-20候选关键词、筛选论文

# 定义同义词合并映射表
synonym_map = {
    'large language model': 'large language models',
    'llm': 'large language models',
    'llms': 'large language models',
    'large language models (llms)': 'large language models',
    'large language model (llm)': 'large language models',
    'large language models(llms)': 'large language models',
    'diffusion model': 'diffusion models',
    'diffusion': 'diffusion models',
    'graph neural network': 'graph neural networks',
    'gnn': 'graph neural networks',
    'gnns': 'graph neural networks',
    'graph neural networks (gnns)': 'graph neural networks',
    'transformer': 'transformers',
    'language model': 'language models',
    'generative model': 'generative models',
    'vision-language model': 'vision-language models',
    'multimodal large language model': 'multimodal large language models',
    'multi-modal large language model': 'multimodal large language models',
    'multi-modal large language models': 'multimodal large language models',
    'multimodal large language models (mllms)': 'multimodal large language models',
    'time series forecasting': 'time series analysis',
    'time-series forecasting': 'time series analysis',
    'time series prediction': 'time series analysis',
    'time-series prediction': 'time series analysis',
    'time-series': 'time series analysis',
    'time-series analysis': 'time series analysis',
}

def clean_keywords(kw_str):
    """清洗关键词：分割、转小写、去空格、同义词合并、去重"""
    if pd.isna(kw_str) or not kw_str.strip():
        return []
    # 按分号分割
    kws = [kw.strip().lower() for kw in kw_str.split(';')]
    # 同义词合并
    kws = [synonym_map.get(kw, kw) for kw in kws if kw]
    # 去重并保持顺序
    seen = set()
    result = []
    for kw in kws:
        if kw not in seen:
            seen.add(kw)
            result.append(kw)
    return result

# 清洗所有论文的关键词
df_kw = df_kw.copy()
df_kw['keywords_clean'] = df_kw['keywords'].apply(clean_keywords)

# 统计每个关键词在所有论文中的出现次数
kw_counter = Counter()
for kw_list in df_kw['keywords_clean']:
    kw_counter.update(kw_list)

print(f'清洗后不重复关键词总数: {len(kw_counter)}')

# 选取出现频次最高的20个关键词作为候选关键词
# 注意：需要排除与status相关的词，候选关键词仅包括主题关键词
candidate_keywords = [kw for kw, _ in kw_counter.most_common(20)]

print(f'\n候选关键词（Top-20）及其出现频次:')
for i, (kw, cnt) in enumerate(kw_counter.most_common(20), 1):
    print(f'  {i:2d}. {kw}: {cnt}')

# 仅保留至少包含1个候选关键词的论文
candidate_set = set(candidate_keywords)
def has_candidate(kw_list):
    return any(kw in candidate_set for kw in kw_list)

mask = df_kw['keywords_clean'].apply(has_candidate)
df_filtered = df_kw[mask].reset_index(drop=True)

n_filtered = len(df_filtered)
n_accept_filtered = df_filtered['status'].sum()
accept_ratio_filtered = n_accept_filtered / n_filtered

print(f'\n筛选后论文数量: {n_filtered}')
print(f'已接收论文数量: {n_accept_filtered}')
print(f'录用比率: {accept_ratio_filtered:.4f} ({accept_ratio_filtered*100:.2f}%)')
print(f'相比筛选前(接收率 {accept_ratio*100:.2f}%)的变化: {(accept_ratio_filtered-accept_ratio)*100:+.2f}%')


清洗后不重复关键词总数: 20722

候选关键词（Top-20）及其出现频次:
   1. large language models: 2504
   2. reinforcement learning: 976
   3. diffusion models: 699
   4. benchmark: 389
   5. reasoning: 333
   6. representation learning: 281
   7. generative models: 267
   8. transformers: 264
   9. graph neural networks: 254
  10. interpretability: 249
  11. deep learning: 243
  12. multimodal large language models: 204
  13. vision-language models: 181
  14. language models: 177
  15. federated learning: 168
  16. flow matching: 166
  17. time series analysis: 147
  18. optimization: 146
  19. evaluation: 142
  20. continual learning: 138

筛选后论文数量: 6420
已接收论文数量: 2408
录用比率: 0.3751 (37.51%)
相比筛选前(接收率 37.81%)的变化: -0.30%


In [4]:
# Task 1d: 构建事务和项集索引字典

# 项集索引字典：候选关键词 + 录用状态
# 录用状态用 'accept' 和 'reject' 表示
ind2val = {}
val2ind = {}

# 首先添加候选关键词
for i, kw in enumerate(candidate_keywords):
    ind2val[i] = kw
    val2ind[kw] = i

# 然后添加录用状态项
status_idx_accept = len(candidate_keywords)
status_idx_reject = len(candidate_keywords) + 1
ind2val[status_idx_accept] = 'accept'
ind2val[status_idx_reject] = 'reject'
val2ind['accept'] = status_idx_accept
val2ind['reject'] = status_idx_reject

print('项集索引字典 (ind2val):')
for idx, name in ind2val.items():
    print(f'  {idx}: {name}')

# 为每篇论文构建事务（候选关键词 + 录用状态 的索引集合）
transactions = []
for _, row in df_filtered.iterrows():
    items = set()
    for kw in row['keywords_clean']:
        if kw in candidate_set:
            items.add(val2ind[kw])
    # 添加录用状态
    if row['status'] == 1:
        items.add(val2ind['accept'])
    else:
        items.add(val2ind['reject'])
    transactions.append(items)

print(f'\n事务总数: {len(transactions)}')
print(f'\n示例事务 (前5条):')
for i, t in enumerate(transactions[:5]):
    items_readable = [ind2val[idx] for idx in sorted(t)]
    print(f'  {i+1}: {items_readable}')


项集索引字典 (ind2val):
  0: large language models
  1: reinforcement learning
  2: diffusion models
  3: benchmark
  4: reasoning
  5: representation learning
  6: generative models
  7: transformers
  8: graph neural networks
  9: interpretability
  10: deep learning
  11: multimodal large language models
  12: vision-language models
  13: language models
  14: federated learning
  15: flow matching
  16: time series analysis
  17: optimization
  18: evaluation
  19: continual learning
  20: accept
  21: reject

事务总数: 6420

示例事务 (前5条):
  1: ['large language models', 'reject']
  2: ['reinforcement learning', 'reject']
  3: ['representation learning', 'reject']
  4: ['reinforcement learning', 'accept']
  5: ['federated learning', 'reject']


In [5]:
# Task 2a: 实现Apriori算法，分别以最小支持度0.02和0.05进行频繁项集挖掘

def apriori(transactions, min_support, n_transactions):
    """
    Apriori算法实现
    transactions: 事务列表，每个事务是一个item索引的集合
    min_support: 最小支持度阈值（绝对计数或比例）
    n_transactions: 事务总数
    """
    # 如果min_support是比例，转换为绝对计数
    if min_support < 1:
        min_support_count = int(min_support * n_transactions)
    else:
        min_support_count = min_support
    
    # Step 1: 生成频繁1-项集
    item_counts = Counter()
    for t in transactions:
        for item in t:
            item_counts[item] += 1
    
    L = []  # 存储各层频繁项集: L[0]=频繁1-项集, L[1]=频繁2-项集, ...
    L1 = []
    support = {}
    for item, count in item_counts.items():
        if count >= min_support_count:
            itemset = (item,)
            L1.append(itemset)
            support[itemset] = count
    L.append(sorted(L1))
    
    if not L[0]:
        return [], support
    
    # Step 2: 逐层生成频繁k-项集 (k >= 2)
    k = 1
    while L[k-1]:
        k += 1
        # 生成候选k-项集：自连接L[k-2]
        candidates = []
        L_prev = L[k-2]
        for i in range(len(L_prev)):
            for j in range(i+1, len(L_prev)):
                itemset_i = L_prev[i]
                itemset_j = L_prev[j]
                # 前k-2项相同时可以连接
                if itemset_i[:k-2] == itemset_j[:k-2]:
                    # 连接：取前k-2项 + 两个不同的最后一项
                    new_itemset = tuple(sorted(set(itemset_i) | set(itemset_j)))
                    if len(new_itemset) == k:
                        # 剪枝：所有k-1子集必须是频繁的
                        is_valid = True
                        for subset in combinations(new_itemset, k-1):
                            if subset not in support:
                                is_valid = False
                                break
                        if is_valid:
                            candidates.append(new_itemset)
        
        # 去重候选集
        candidates = list(set(candidates))
        
        if not candidates:
            break
        
        # 计数支持度
        cand_counts = Counter()
        for t in transactions:
            t_set = set(t)
            for cand in candidates:
                if all(item in t_set for item in cand):
                    cand_counts[cand] += 1
        
        # 筛选频繁k-项集
        Lk = []
        for cand, count in cand_counts.items():
            if count >= min_support_count:
                Lk.append(cand)
                support[cand] = count
        
        if Lk:
            L.append(sorted(Lk))
        else:
            break
    
    return L, support

n_transactions = len(transactions)

# ===== 以最小支持度0.02进行挖掘 =====
L_002, support_002 = apriori(transactions, 0.02, n_transactions)

print('='*60)
print(f'最小支持度 = 0.02 (绝对计数: {int(0.02*n_transactions)})')
print('='*60)
for k, Lk in enumerate(L_002):
    print(f'\n频繁{k+1}-项集数量: {len(Lk)}')
    for itemset in Lk[:10]:
        items_readable = [ind2val[idx] for idx in itemset]
        sup = support_002[itemset] / n_transactions
        print(f'  {items_readable}  (支持度: {sup:.4f})')
    if len(Lk) > 10:
        print(f'  ... 及另外 {len(Lk)-10} 个')

if len(L_002) >= 3 and L_002[2]:
    print(f'\n频繁3-项集共{len(L_002[2])}个:')
    for itemset in L_002[2]:
        items_readable = [ind2val[idx] for idx in itemset]
        sup = support_002[itemset] / n_transactions
        print(f'  {items_readable}  (支持度: {sup:.4f})')
else:
    print(f'\n未发现频繁3-项集 (最小支持度=0.02)')

# ===== 以最小支持度0.05进行挖掘 =====
L_005, support_005 = apriori(transactions, 0.05, n_transactions)

print('\n' + '='*60)
print(f'最小支持度 = 0.05 (绝对计数: {int(0.05*n_transactions)})')
print('='*60)
for k, Lk in enumerate(L_005):
    print(f'\n频繁{k+1}-项集数量: {len(Lk)}')
    for itemset in Lk[:10]:
        items_readable = [ind2val[idx] for idx in itemset]
        sup = support_005[itemset] / n_transactions
        print(f'  {items_readable}  (支持度: {sup:.4f})')
    if len(Lk) > 10:
        print(f'  ... 及另外 {len(Lk)-10} 个')

if len(L_005) >= 3 and L_005[2]:
    print(f'\n频繁3-项集共{len(L_005[2])}个:')
    for itemset in L_005[2]:
        items_readable = [ind2val[idx] for idx in itemset]
        sup = support_005[itemset] / n_transactions
        print(f'  {items_readable}  (支持度: {sup:.4f})')
else:
    print(f'\n未发现频繁3-项集 (最小支持度=0.05)')


最小支持度 = 0.02 (绝对计数: 128)

频繁1-项集数量: 22
  ['large language models']  (支持度: 0.3900)
  ['reinforcement learning']  (支持度: 0.1520)
  ['diffusion models']  (支持度: 0.1089)
  ['benchmark']  (支持度: 0.0606)
  ['reasoning']  (支持度: 0.0519)
  ['representation learning']  (支持度: 0.0438)
  ['generative models']  (支持度: 0.0416)
  ['transformers']  (支持度: 0.0411)
  ['graph neural networks']  (支持度: 0.0396)
  ['interpretability']  (支持度: 0.0388)
  ... 及另外 12 个

频繁2-项集数量: 18
  ['large language models', 'reinforcement learning']  (支持度: 0.0391)
  ['large language models', 'reasoning']  (支持度: 0.0299)
  ['large language models', 'accept']  (支持度: 0.1435)
  ['large language models', 'reject']  (支持度: 0.2466)
  ['reinforcement learning', 'accept']  (支持度: 0.0634)
  ['reinforcement learning', 'reject']  (支持度: 0.0886)
  ['diffusion models', 'accept']  (支持度: 0.0456)
  ['diffusion models', 'reject']  (支持度: 0.0632)
  ['benchmark', 'accept']  (支持度: 0.0246)
  ['benchmark', 'reject']  (支持度: 0.0360)
  ... 及另外 8 个

频繁3-项集数量: 1
  

In [6]:
# Task 2b: 将20个候选关键词合并为6个主题类别，重新构建事务并挖掘频繁项集

# 6个主题类别映射表
category_map = {
    # LLM
    'large language models': 'LLM',
    'language models': 'LLM',
    'reasoning': 'LLM',
    # MLLM
    'multimodal large language models': 'MLLM',
    'vision-language models': 'MLLM',
    # Generative
    'diffusion models': 'Generative',
    'generative models': 'Generative',
    'flow matching': 'Generative',
    # RL
    'reinforcement learning': 'RL',
    # Evaluation
    'benchmark': 'Evaluation',
    'evaluation': 'Evaluation',
    'interpretability': 'Evaluation',
    # ML_Foundation
    'representation learning': 'ML_Foundation',
    'deep learning': 'ML_Foundation',
    'transformers': 'ML_Foundation',
    'graph neural networks': 'ML_Foundation',
    'optimization': 'ML_Foundation',
    'federated learning': 'ML_Foundation',
    'time series analysis': 'ML_Foundation',
    'continual learning': 'ML_Foundation',
}

# 检查：确保所有20个候选关键词都已映射
unmapped = [kw for kw in candidate_keywords if kw not in category_map]
if unmapped:
    print(f'警告：以下关键词未映射到主题类别: {unmapped}')
else:
    print('所有20个候选关键词均已成功映射到6个主题类别。')

# 列出主题类别及其包含的候选关键词
print('\n主题类别映射:')
cat_to_kws = {}
for kw, cat in category_map.items():
    if cat not in cat_to_kws:
        cat_to_kws[cat] = []
    cat_to_kws[cat].append(kw)
for cat, kws in cat_to_kws.items():
    print(f'  {cat}: {", ".join(kws)}')

# 重新构建事务：候选关键词替换为主题类别（同一论文中同主题去重）
theme_transactions = []
theme_ind2val = {}
theme_val2ind = {}

# 6个主题类别 + 录用状态 (accept/reject)
theme_names = ['LLM', 'MLLM', 'Generative', 'RL', 'Evaluation', 'ML_Foundation']
for i, theme in enumerate(theme_names):
    theme_ind2val[i] = theme
    theme_val2ind[theme] = i

theme_ind2val[len(theme_names)] = 'accept'
theme_ind2val[len(theme_names)+1] = 'reject'
theme_val2ind['accept'] = len(theme_names)
theme_val2ind['reject'] = len(theme_names)+1

print('\n主题项集索引字典:')
for idx, name in theme_ind2val.items():
    print(f'  {idx}: {name}')

# 构建主题事务
for _, row in df_filtered.iterrows():
    items = set()
    for kw in row['keywords_clean']:
        if kw in candidate_set and kw in category_map:
            theme = category_map[kw]
            items.add(theme_val2ind[theme])
    # 添加录用状态
    if row['status'] == 1:
        items.add(theme_val2ind['accept'])
    else:
        items.add(theme_val2ind['reject'])
    theme_transactions.append(items)

print(f'\n主题事务总数: {len(theme_transactions)}')
print(f'\n示例主题事务 (前5条):')
for i, t in enumerate(theme_transactions[:5]):
    items_readable = [theme_ind2val[idx] for idx in sorted(t)]
    print(f'  {i+1}: {items_readable}')

# 以最小支持度0.02进行频繁项集挖掘
n_theme_trans = len(theme_transactions)
L_theme, support_theme = apriori(theme_transactions, 0.02, n_theme_trans)

print('\n' + '='*60)
print(f'主题级频繁项集挖掘 (最小支持度 = 0.02, 绝对计数: {int(0.02*n_theme_trans)})')
print('='*60)

# 统计每个主题的出现频次
theme_counts = Counter()
for t in theme_transactions:
    for item in t:
        if item < len(theme_names):  # 仅主题项
            theme_counts[item] += 1

print('\n各主题出现频次:')
for idx, name in theme_ind2val.items():
    if idx < len(theme_names):
        cnt = theme_counts.get(idx, 0)
        print(f'  {name}: {cnt} ({cnt/n_theme_trans*100:.1f}%)')

for k, Lk in enumerate(L_theme):
    print(f'\n频繁{k+1}-项集数量: {len(Lk)}')
    for itemset in Lk:
        items_readable = [theme_ind2val[idx] for idx in itemset]
        sup = support_theme[itemset] / n_theme_trans
        print(f'  {items_readable}  (支持度: {sup:.4f})')

所有20个候选关键词均已成功映射到6个主题类别。

主题类别映射:
  LLM: large language models, language models, reasoning
  MLLM: multimodal large language models, vision-language models
  Generative: diffusion models, generative models, flow matching
  RL: reinforcement learning
  Evaluation: benchmark, evaluation, interpretability
  ML_Foundation: representation learning, deep learning, transformers, graph neural networks, optimization, federated learning, time series analysis, continual learning

主题项集索引字典:
  0: LLM
  1: MLLM
  2: Generative
  3: RL
  4: Evaluation
  5: ML_Foundation
  6: accept
  7: reject

主题事务总数: 6420

示例主题事务 (前5条):
  1: ['LLM', 'reject']
  2: ['RL', 'reject']
  3: ['ML_Foundation', 'reject']
  4: ['RL', 'accept']
  5: ['ML_Foundation', 'reject']

主题级频繁项集挖掘 (最小支持度 = 0.02, 绝对计数: 128)

各主题出现频次:
  LLM: 2795 (43.5%)
  MLLM: 384 (6.0%)
  Generative: 1017 (15.8%)
  RL: 976 (15.2%)
  Evaluation: 738 (11.5%)
  ML_Foundation: 1518 (23.6%)

频繁1-项集数量: 8
  ['LLM']  (支持度: 0.4354)
  ['MLLM']  (支持度: 0.0598)
 

In [8]:
# Task 3a: 基于Task 2b得到的频繁项集，提取形如 X→{accept} 的关联规则 (min_confidence=0.4)

def generate_rules(L, support, n_transactions, min_confidence, target_consequent, val2ind, ind2val):
    """
    从频繁项集中提取关联规则
    L: 频繁项集列表
    support: 支持度字典
    n_transactions: 事务总数
    min_confidence: 最小置信度阈值
    target_consequent: 目标后件（如 accept 的索引）
    """
    rules = []
    for k in range(1, len(L)):
        for itemset in L[k]:
            itemset_set = set(itemset)
            if target_consequent not in itemset_set:
                continue
            # 后件为 {target_consequent}
            consequent = (target_consequent,)
            # 前件为 itemset - 后件
            antecedent = tuple(sorted(itemset_set - {target_consequent}))
            if not antecedent:
                continue
            # 计算各项指标
            support_union = support[itemset]
            support_antecedent = support[antecedent]
            confidence = support_union / support_antecedent
            if confidence >= min_confidence:
                support_consequent = support.get(consequent, 0)
                lift = confidence / (support_consequent / n_transactions)
                rules.append({
                    'antecedent': antecedent,
                    'consequent': consequent,
                    'support': support_union / n_transactions,
                    'confidence': confidence,
                    'lift': lift,
                })
    # 按提升度降序排列
    rules.sort(key=lambda r: r['lift'], reverse=True)
    return rules

target = theme_val2ind['accept']
rules = generate_rules(L_theme, support_theme, n_theme_trans, 0.4, target, theme_val2ind, theme_ind2val)

print('='*60)
print(f'关联规则提取: X -> {{accept}} (min_confidence = 0.4)')
print(f'共提取到 {len(rules)} 条规则')
print('='*60)

print(f'\n{"序号":<5} {"前件(X)":<40} {"支持度":<10} {"置信度":<10} {"提升度":<10}')
print('-'*75)
for i, rule in enumerate(rules, 1):
    ante_readable = ', '.join([theme_ind2val[idx] for idx in rule['antecedent']])
    print(f'{i:<5} {ante_readable:<40} {rule["support"]:<10.4f} {rule["confidence"]:<10.4f} {rule["lift"]:<10.4f}')

print(f'\n整体论文录用率: {df_filtered["status"].mean():.4f}')


关联规则提取: X -> {accept} (min_confidence = 0.4)
共提取到 4 条规则

序号    前件(X)                                    支持度        置信度        提升度       
---------------------------------------------------------------------------
1     LLM, RL                                  0.0215     0.4808     1.2820    
2     MLLM                                     0.0259     0.4323     1.1525    
3     Generative                               0.0668     0.4218     1.1246    
4     RL                                       0.0634     0.4170     1.1118    

整体论文录用率: 0.3751
